In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import pandas as pd

input_folder = "/content/drive/MyDrive/AI_NIDS_Dataset/Task4_Output"

X_train = pd.read_parquet(os.path.join(input_folder, "X_train.parquet"))
X_val = pd.read_parquet(os.path.join(input_folder, "X_val.parquet"))
X_test = pd.read_parquet(os.path.join(input_folder, "X_test.parquet"))

y_train = pd.read_parquet(os.path.join(input_folder, "y_train.parquet"))["Label"]
y_val = pd.read_parquet(os.path.join(input_folder, "y_val.parquet"))["Label"]
y_test = pd.read_parquet(os.path.join(input_folder, "y_test.parquet"))["Label"]

with open(os.path.join(input_folder, "class_weights.json"), "r") as f:
    class_weight_dict = json.load(f)

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Testing data:", X_test.shape)
print("Number of classes:", y_train.nunique())
print("Class weights loaded:", len(class_weight_dict))

Training data: (1800848, 78)
Validation data: (385896, 78)
Testing data: (385896, 78)
Number of classes: 15
Class weights loaded: 15


In [ ]:
print("Training data column types:")
print(X_train.dtypes.value_counts())

print("\nNon-numeric columns:")
non_numeric = X_train.select_dtypes(exclude=["number"]).columns.tolist()
print(non_numeric)

print("\nMissing values:", X_train.isnull().sum().sum())
print("Infinite values:", (~X_train.apply(lambda col: col.map(lambda x: abs(x) != float("inf")))).sum().sum())

Training data column types:
int64      54
float64    24
Name: count, dtype: int64

Non-numeric columns:
[]

Missing values: 0
Infinite values: 0


In [ ]:
import importlib.util

libraries = ["lightgbm", "xgboost", "sklearn"]

for library in libraries:
    if importlib.util.find_spec(library):
        print(library, "is installed")
    else:
        print(library, "is NOT installed")

lightgbm is installed
xgboost is installed
sklearn is installed


In [ ]:
import lightgbm as lgb
from lightgbm import LGBMClassifier

lightgbm_model = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    class_weight=class_weight_dict,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lightgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=50)
    ]
)

print("LightGBM training completed!")
print("Best iteration:", lightgbm_model.best_iteration_)

Training until validation scores don't improve for 30 rounds
[50]	valid_0's multi_logloss: 14.7626
Early stopping, best iteration is:
[23]	valid_0's multi_logloss: 0.0481455
LightGBM training completed!
Best iteration: 23


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_val_pred = lightgbm_model.predict(X_val)

print("LightGBM Validation Accuracy:",
      accuracy_score(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, zero_division=0))

LightGBM Validation Accuracy: 0.9974889607562659

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    322035
                       Bot       0.66      0.98      0.79       292
                      DDoS       1.00      1.00      1.00     19202
             DoS GoldenEye       0.99      1.00      0.99      1543
                  DoS Hulk       1.00      1.00      1.00     25927
          DoS Slowhttptest       0.95      0.99      0.97       784
             DoS slowloris       0.97      0.98      0.98       808
               FTP-Patator       1.00      1.00      1.00       889
                Heartbleed       0.00      0.00      0.00         2
              Infiltration       0.20      0.20      0.20         5
                  PortScan       0.99      1.00      0.99     13604
               SSH-Patator       0.98      0.99      0.99       483
  Web Attack - Brute Force       0.57     

In [ ]:
import json
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

lightgbm_results = {
    "Model": "LightGBM",
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Macro Precision": precision_score(y_val, y_val_pred, average="macro", zero_division=0),
    "Macro Recall": recall_score(y_val, y_val_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_val, y_val_pred, average="macro", zero_division=0),
    "Weighted F1": f1_score(y_val, y_val_pred, average="weighted", zero_division=0)
}

results_folder = "/content/drive/MyDrive/AI_NIDS_Dataset/Task5_Output"
os.makedirs(results_folder, exist_ok=True)

with open(os.path.join(results_folder, "lightgbm_results.json"), "w") as f:
    json.dump(lightgbm_results, f, indent=4)

print(lightgbm_results)
print("LightGBM results saved successfully!")

{'Model': 'LightGBM', 'Accuracy': 0.9974889607562659, 'Macro Precision': 0.7082103740939096, 'Macro Recall': 0.7659058450049502, 'Macro F1': 0.7297536518255784, 'Weighted F1': 0.9978686432841085}
LightGBM results saved successfully!


In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

sample_weights = y_train.map(class_weight_dict).values

xgboost_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30
)

xgboost_model.fit(
    X_train,
    y_train_encoded,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val_encoded)],
    verbose=50
)

print("XGBoost training completed!")
print("Best iteration:", xgboost_model.best_iteration)

[0]	validation_0-mlogloss:0.83008
[50]	validation_0-mlogloss:0.01120
[100]	validation_0-mlogloss:0.00519
[150]	validation_0-mlogloss:0.00504
[179]	validation_0-mlogloss:0.00508
XGBoost training completed!
Best iteration: 149


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_xgb_pred_encoded = xgboost_model.predict(X_val)
y_xgb_pred = label_encoder.inverse_transform(
    y_xgb_pred_encoded.astype(int)
)

print("XGBoost Validation Accuracy:",
      accuracy_score(y_val, y_xgb_pred))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_xgb_pred,
    zero_division=0
))

XGBoost Validation Accuracy: 0.9986732176544976

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    322035
                       Bot       0.70      0.97      0.81       292
                      DDoS       1.00      1.00      1.00     19202
             DoS GoldenEye       0.99      1.00      0.99      1543
                  DoS Hulk       1.00      1.00      1.00     25927
          DoS Slowhttptest       0.98      0.99      0.99       784
             DoS slowloris       0.99      0.99      0.99       808
               FTP-Patator       1.00      1.00      1.00       889
                Heartbleed       1.00      0.50      0.67         2
              Infiltration       1.00      0.80      0.89         5
                  PortScan       0.99      1.00      0.99     13604
               SSH-Patator       1.00      1.00      1.00       483
  Web Attack - Brute Force       0.71      

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import json
import os

xgboost_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_val, y_xgb_pred),
    "Macro Precision": precision_score(
        y_val, y_xgb_pred, average="macro", zero_division=0
    ),
    "Macro Recall": recall_score(
        y_val, y_xgb_pred, average="macro", zero_division=0
    ),
    "Macro F1": f1_score(
        y_val, y_xgb_pred, average="macro", zero_division=0
    ),
    "Weighted F1": f1_score(
        y_val, y_xgb_pred, average="weighted", zero_division=0
    )
}

with open(os.path.join(results_folder, "xgboost_results.json"), "w") as f:
    json.dump(xgboost_results, f, indent=4)

print(xgboost_results)
print("XGBoost results saved successfully!")

{'Model': 'XGBoost', 'Accuracy': 0.9986732176544976, 'Macro Precision': 0.9157839355293079, 'Macro Recall': 0.8434007109217434, 'Macro F1': 0.8600902914132909, 'Weighted F1': 0.9986878308702698}
XGBoost results saved successfully!


In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=class_weight_dict,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

print("Random Forest training completed!")
print("Number of trees:", len(random_forest_model.estimators_))

Random Forest training completed!
Number of trees: 200


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_rf_pred = random_forest_model.predict(X_val)

print("Random Forest Validation Accuracy:",
      accuracy_score(y_val, y_rf_pred))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_rf_pred,
    zero_division=0
))

Random Forest Validation Accuracy: 0.9983156083504364

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    322035
                       Bot       0.89      0.78      0.83       292
                      DDoS       1.00      1.00      1.00     19202
             DoS GoldenEye       0.99      1.00      1.00      1543
                  DoS Hulk       1.00      1.00      1.00     25927
          DoS Slowhttptest       0.99      0.99      0.99       784
             DoS slowloris       1.00      0.99      0.99       808
               FTP-Patator       1.00      1.00      1.00       889
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.80      0.89         5
                  PortScan       0.99      0.99      0.99     13604
               SSH-Patator       1.00      0.99      0.99       483
  Web Attack - Brute Force       0.75

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import json
import os

random_forest_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_val, y_rf_pred),
    "Macro Precision": precision_score(
        y_val, y_rf_pred, average="macro", zero_division=0
    ),
    "Macro Recall": recall_score(
        y_val, y_rf_pred, average="macro", zero_division=0
    ),
    "Macro F1": f1_score(
        y_val, y_rf_pred, average="macro", zero_division=0
    ),
    "Weighted F1": f1_score(
        y_val, y_rf_pred, average="weighted", zero_division=0
    )
}

with open(os.path.join(results_folder, "random_forest_results.json"), "w") as f:
    json.dump(random_forest_results, f, indent=4)

print(random_forest_results)
print("Random Forest results saved successfully!")

{'Model': 'Random Forest', 'Accuracy': 0.9983156083504364, 'Macro Precision': 0.8704900509669324, 'Macro Recall': 0.8453951071819773, 'Macro F1': 0.8565454114827857, 'Weighted F1': 0.9982912719999543}
Random Forest results saved successfully!


In [ ]:
import pandas as pd
import os

comparison_df = pd.DataFrame([
    lightgbm_results,
    xgboost_results,
    random_forest_results
])

comparison_df = comparison_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

comparison_path = os.path.join(
    results_folder,
    "Model_Comparison.csv"
)

comparison_df.to_csv(comparison_path, index=False)

print("Final Model Comparison:")
display(comparison_df)

print("\nComparison table saved at:", comparison_path)

Final Model Comparison:


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,XGBoost,0.998673,0.915784,0.843401,0.860090,0.998688
1,Random Forest,0.998316,0.870490,0.845395,0.856545,0.998291
2,LightGBM,0.997489,0.708210,0.765906,0.729754,0.997869



Comparison table saved at: /content/drive/MyDrive/AI_NIDS_Dataset/Task5_Output/Model_Comparison.csv


In [ ]:
import joblib
import os

model_folder = os.path.join(results_folder, "Trained_Models")
os.makedirs(model_folder, exist_ok=True)

joblib.dump(
    lightgbm_model,
    os.path.join(model_folder, "LightGBM_Model.pkl")
)

joblib.dump(
    xgboost_model,
    os.path.join(model_folder, "XGBoost_Model.pkl")
)

joblib.dump(
    random_forest_model,
    os.path.join(model_folder, "Random_Forest_Model.pkl")
)

joblib.dump(
    label_encoder,
    os.path.join(model_folder, "Label_Encoder.pkl")
)

print("All three trained models saved successfully!")
print("Model folder:", model_folder)

All three trained models saved successfully!
Model folder: /content/drive/MyDrive/AI_NIDS_Dataset/Task5_Output/Trained_Models


In [ ]:
import os

print("Task 5 Output Files:\n")

for root, dirs, files in os.walk(results_folder):
    for file in files:
        file_path = os.path.join(root, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)

        print(f"{file} - {size_mb:.2f} MB")

Task 5 Output Files:

lightgbm_results.json - 0.00 MB
xgboost_results.json - 0.00 MB
random_forest_results.json - 0.00 MB
Model_Comparison.csv - 0.00 MB
LightGBM_Model.pkl - 1.16 MB
XGBoost_Model.pkl - 7.37 MB
Random_Forest_Model.pkl - 275.35 MB
Label_Encoder.pkl - 0.00 MB
